<a href="https://oscar-defelice.github.io">
<img src="../../img/image.png" height="125" align="right" />
</a>

# TP 03-B — Logistic Regression (SOLUTION)

> **Instructor only — do not distribute.**

Expected val macro F1 milestones:
| Configuration | Val macro F1 |
|---|---|
| LR TF-IDF unigrams, C=1.0 | ~0.887–0.893 |
| LR TF-IDF unigrams, best C | ~0.889–0.895 |
| LR TF-IDF bigrams, best C | ~0.893–0.900 |

Common student mistakes are flagged with `# ⚠️ STUDENT MISTAKE`.


In [ ]:
import re
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
)
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB

warnings.filterwarnings("ignore")
sns.set_style("darkgrid")
SEED = 42
np.random.seed(SEED)
AG_NEWS_LABELS = {1: "World", 2: "Sports", 3: "Business", 4: "Sci/Tech"}
print("Imports OK")


In [ ]:
def preprocess_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[^a-z0-9 ]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def load_ag_news(
    val_size: float = 0.1,
    seed: int = SEED,
) -> tuple:
    raw = load_dataset("ag_news")
    rng = np.random.default_rng(seed)

    train_texts_raw  = [r["text"]  for r in raw["train"]]
    train_labels_raw = [r["label"] for r in raw["train"]]
    test_texts  = [preprocess_text(r["text"])  for r in raw["test"]]
    test_labels = [r["label"]                  for r in raw["test"]]

    n = len(train_texts_raw)
    idx   = rng.permutation(n)
    split = int(n * (1 - val_size))
    train_idx, val_idx = idx[:split], idx[split:]

    train_texts  = [preprocess_text(train_texts_raw[i]) for i in train_idx]
    train_labels = [train_labels_raw[i]                 for i in train_idx]
    val_texts    = [preprocess_text(train_texts_raw[i]) for i in val_idx]
    val_labels   = [train_labels_raw[i]                 for i in val_idx]

    return train_texts, val_texts, test_texts, train_labels, val_labels, test_labels


train_texts, val_texts, test_texts, train_labels, val_labels, test_labels = load_ag_news()
label_names = [AG_NEWS_LABELS[i + 1] for i in range(4)]
print(f"Train: {len(train_texts):,}  Val: {len(val_texts):,}  Test: {len(test_texts):,}")


## Part 2 — Baseline Logistic Regression

In [ ]:
def build_lr_pipeline(
    C: float = 1.0,
    penalty: str = "l2",
    vectorizer: str = "tfidf",
    max_features: int | None = 50_000,
    ngram_range: tuple = (1, 1),
    max_iter: int = 1000,
) -> Pipeline:
    # ⚠️ STUDENT MISTAKE: using solver='lbfgs' for L1 — lbfgs does not support L1.
    # Use 'saga' for L1, 'lbfgs' for L2 / no penalty.
    if vectorizer == "tfidf":
        vec = TfidfVectorizer(
            max_features=max_features,
            ngram_range=ngram_range,
            sublinear_tf=True,
        )
    elif vectorizer == "count":
        vec = CountVectorizer(max_features=max_features, ngram_range=ngram_range)
    else:
        raise ValueError(f"Unknown vectorizer: {vectorizer!r}. Use 'tfidf' or 'count'.")

    solver = "saga" if penalty == "l1" else "lbfgs"

    clf = LogisticRegression(
        C=C,
        penalty=penalty,
        solver=solver,
        multi_class="multinomial",
        max_iter=max_iter,
        random_state=SEED,
    )
    return Pipeline([("vec", vec), ("clf", clf)])


def evaluate_classifier(
    y_true,
    y_pred,
    label_names,
    title: str = "Evaluation",
) -> dict:
    report = classification_report(
        y_true, y_pred, target_names=label_names, output_dict=True
    )
    print(f"\n{'='*60}\n  {title}\n{'='*60}")
    print(classification_report(y_true, y_pred, target_names=label_names))

    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay(cm, display_labels=label_names).plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

    return {
        "accuracy":  round(report["accuracy"], 4),
        "macro_f1":  round(report["macro avg"]["f1-score"], 4),
    }


In [ ]:
pipe_lr = build_lr_pipeline(C=1.0)
pipe_lr.fit(train_texts, train_labels)
val_pred_lr = pipe_lr.predict(val_texts)

results_lr = evaluate_classifier(
    val_labels, val_pred_lr, label_names,
    title="LR (TF-IDF, C=1.0, L2) — validation",
)


### Instructor notes — Part 2

Expected val macro F1: **~0.889** (C=1.0, TF-IDF unigrams, L2).

Business↔Sci/Tech is the most confused pair (tech company earnings stories straddle both).
World↔Business also appears (geopolitical trade stories).
Sports is cleanest — very domain-specific vocabulary.

**Compare to NB TF-IDF (TP-A):** LR should be +1–3% F1, especially on Business/Sci/Tech.


## Part 3 — Regularisation sweep

In [ ]:
def regularisation_sweep(
    C_values: list,
    train_texts: list,
    train_labels: list,
    val_texts: list,
    val_labels: list,
    penalty: str = "l2",
) -> pd.DataFrame:
    # ⚠️ STUDENT MISTAKE: only tracking val score — they miss the train score
    # needed to diagnose overfitting. Always track both.
    rows = []
    for C in C_values:
        pipe = build_lr_pipeline(C=C, penalty=penalty)
        pipe.fit(train_texts, train_labels)

        train_pred = pipe.predict(train_texts)
        val_pred   = pipe.predict(val_texts)

        rows.append({
            "C":             C,
            "val_accuracy":  round(classification_report(val_labels,   val_pred,   output_dict=True)["accuracy"],             4),
            "val_macro_f1":  round(f1_score(val_labels,   val_pred,   average="macro"), 4),
            "train_macro_f1":round(f1_score(train_labels, train_pred, average="macro"), 4),
        })
    return pd.DataFrame(rows)


C_values = [0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0]
sweep_df = regularisation_sweep(
    C_values, train_texts, train_labels, val_texts, val_labels, penalty="l2"
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogx(sweep_df["C"], sweep_df["val_macro_f1"],    marker="o", label="val macro F1")
ax.semilogx(sweep_df["C"], sweep_df["train_macro_f1"],  marker="s", linestyle="--", label="train macro F1")
ax.set_xlabel("C (inverse regularisation)")
ax.set_ylabel("macro F1")
ax.set_title("Logistic Regression — L2 regularisation sweep")
ax.legend()
plt.tight_layout()
plt.show()

best_C = sweep_df.loc[sweep_df["val_macro_f1"].idxmax(), "C"]
print(f"Best C = {best_C}")
print(sweep_df.to_string(index=False))


### Instructor notes — Part 3 (regularisation)

Typical results:
- Very small C (0.001): underfitting, val F1 ~0.70–0.78
- Best C: usually **1.0–5.0**, val F1 ~0.890–0.895
- Large C (50): train F1 approaches 1.0, val F1 drops slightly — classic overfitting gap

The train/val gap widens visibly at C >= 10. Students should identify C ~1 as the sweet spot.


In [ ]:
def compare_penalties(
    C: float,
    train_texts: list,
    train_labels: list,
    val_texts: list,
    val_labels: list,
) -> pd.DataFrame:
    rows = []
    for penalty in ["l1", "l2"]:
        pipe = build_lr_pipeline(C=C, penalty=penalty)
        pipe.fit(train_texts, train_labels)
        val_pred = pipe.predict(val_texts)

        coef = pipe["clf"].coef_          # shape (n_classes, vocab)
        n_zero    = int(np.sum(coef == 0.0))
        sparsity  = round(n_zero / coef.size, 4)

        rows.append({
            "penalty":        penalty,
            "val_macro_f1":   round(f1_score(val_labels, val_pred, average="macro"), 4),
            "n_zero_weights": n_zero,
            "sparsity":       sparsity,
        })
    return pd.DataFrame(rows)


penalty_df = compare_penalties(
    best_C, train_texts, train_labels, val_texts, val_labels
)
print(penalty_df.to_string(index=False))


### Instructor notes — Part 3 (L1 vs L2)

Expected sparsity: L1 typically zeros out **40–70%** of the weight matrix at C=1.
L2 is almost always non-sparse (near 0% zeros).

Val F1 is usually within 0.5–1% between L1 and L2. L2 is marginally more stable.

Interesting teaching point: L1 performs implicit feature selection — the effective
vocabulary the model uses can be 30–60% smaller than the TF-IDF vocabulary.


## Part 4 — Feature analysis: learned weights

In [ ]:
def top_weights_per_class(
    pipeline: Pipeline,
    label_names: list,
    n: int = 20,
) -> dict:
    vocab = np.array(pipeline["vec"].get_feature_names_out())
    coef  = pipeline["clf"].coef_   # (n_classes, vocab_size)

    result = {}
    for cls_idx, cls_name in enumerate(label_names):
        weights = coef[cls_idx]
        top_pos_idx = np.argsort(weights)[-n:][::-1]
        top_neg_idx = np.argsort(weights)[:n]

        rows = []
        for idx in top_pos_idx:
            rows.append({"word": vocab[idx], "weight": round(weights[idx], 4), "direction": "+"})
        for idx in top_neg_idx:
            rows.append({"word": vocab[idx], "weight": round(weights[idx], 4), "direction": "-"})

        result[cls_name] = pd.DataFrame(rows)
    return result


def plot_top_weights(
    weights: dict,
    n_plot: int = 15,
) -> None:
    fig, axes = plt.subplots(1, len(weights), figsize=(5 * len(weights), 6))
    colors = {"+" : "#3a86ff", "-": "#ff595e"}

    for ax, (cls_name, df) in zip(axes, weights.items()):
        pos = df[df["direction"] == "+"].head(n_plot // 2).sort_values("weight")
        neg = df[df["direction"] == "-"].head(n_plot // 2).sort_values("weight", ascending=False)
        combined = pd.concat([neg, pos]).reset_index(drop=True)

        bar_colors = [colors[d] for d in combined["direction"]]
        ax.barh(combined["word"], combined["weight"], color=bar_colors)
        ax.axvline(0, color="black", linewidth=0.8)
        ax.set_title(cls_name, fontsize=12, fontweight="bold")
        ax.set_xlabel("weight")

    plt.suptitle("Top LR weights per class (+ positive, - negative)", fontsize=13)
    plt.tight_layout()
    plt.show()


pipe_best = build_lr_pipeline(C=best_C)
pipe_best.fit(train_texts, train_labels)
weights = top_weights_per_class(pipe_best, label_names, n=20)

for name, df in weights.items():
    print(f"\n{name} — top 10 positive:")
    print(df[df["direction"] == "+"].head(10).to_string(index=False))

plot_top_weights(weights)


In [ ]:
def compare_nb_lr_features(
    nb_features: dict,
    lr_weights: dict,
    class_name: str,
    top_k: int = 20,
) -> pd.DataFrame:
    nb_df  = nb_features[class_name]
    lr_df  = lr_weights[class_name]

    nb_top = set(nb_df.head(top_k)["word"].tolist())
    lr_top = set(lr_df[lr_df["direction"] == "+"].head(top_k)["word"].tolist())

    all_words = nb_top | lr_top
    rows = []
    for word in all_words:
        in_nb    = word in nb_top
        in_lr    = word in lr_top
        nb_score = nb_df.loc[nb_df["word"] == word, "score"].values
        lr_wt    = lr_df.loc[lr_df["word"] == word, "weight"].values
        rows.append({
            "word":      word,
            "in_nb":     in_nb,
            "in_lr":     in_lr,
            "nb_score":  round(float(nb_score[0]), 4) if len(nb_score) else float("nan"),
            "lr_weight": round(float(lr_wt[0]),    4) if len(lr_wt)    else float("nan"),
        })
    df = pd.DataFrame(rows)
    df["both"] = df["in_nb"] & df["in_lr"]
    df = df.sort_values(["both", "lr_weight"], ascending=[False, False])
    overlap_pct = round(len(nb_top & lr_top) / top_k * 100, 1)
    print(f"{class_name} — overlap between top-{top_k} NB and LR features: {overlap_pct}%")
    return df


# ── Re-fit NB to get nb_features ─────────────────────────────────────────────
def most_informative_features(pipeline, label_names, n=20):
    vocab  = np.array(pipeline["vec"].get_feature_names_out())
    log_p  = pipeline["clf"].feature_log_prob_   # (n_classes, vocab)
    scores = log_p - log_p.mean(axis=0)          # deviation from mean
    result = {}
    for cls_idx, cls_name in enumerate(label_names):
        top_idx = np.argsort(scores[cls_idx])[-n:][::-1]
        result[cls_name] = pd.DataFrame({
            "word":  vocab[top_idx],
            "score": scores[cls_idx][top_idx].round(4),
        })
    return result


nb_pipe = Pipeline([
    ("vec", TfidfVectorizer(max_features=50_000, sublinear_tf=True)),
    ("clf", MultinomialNB(alpha=0.1)),
])
nb_pipe.fit(train_texts, train_labels)
nb_features = most_informative_features(nb_pipe, label_names, n=20)

comparison = compare_nb_lr_features(nb_features, weights, class_name="Sports")
print(comparison.to_string(index=False))


### Instructor notes — Part 4

**Expected top positive weights:**
- World: "iraq", "troops", "government", "minister", "killed"
- Sports: "league", "cup", "coach", "injury", "season", "champion"
- Business: "shares", "quarter", "revenue", "earnings", "forecast"
- Sci/Tech: "software", "linux", "processor", "broadband", "search"

**Negative weights** are equally telling — for Sports, negative weights include
financial vocabulary ("revenue", "shares") and geopolitical terms.

**NB vs LR overlap:** typically 60–80% for Sports and World (distinctive vocab),
50–65% for Business/Sci/Tech (overlapping vocabulary).

**Key pedagogical point:** LR weights capture global discriminativity (word vs all other classes).
NB log-probs are class-conditional — they can over-weight rare but class-specific words.


## Part 5 — Full ablation

In [ ]:
def full_ablation(
    train_texts: list,
    train_labels: list,
    val_texts: list,
    val_labels: list,
    best_C: float,
    best_nb_alpha: float = 0.1,
) -> pd.DataFrame:
    configs = [
        ("NB       | count unigrams",   Pipeline([("vec", CountVectorizer(max_features=50_000)),                                         ("clf", MultinomialNB(alpha=best_nb_alpha))])),
        ("NB       | tfidf unigrams",   Pipeline([("vec", TfidfVectorizer(max_features=50_000, sublinear_tf=True)),                      ("clf", MultinomialNB(alpha=best_nb_alpha))])),
        ("NB       | tfidf bigrams",    Pipeline([("vec", TfidfVectorizer(max_features=50_000, sublinear_tf=True, ngram_range=(1,2))),   ("clf", MultinomialNB(alpha=best_nb_alpha))])),
        ("LR       | tfidf unigrams",   build_lr_pipeline(C=best_C)),
        ("LR       | tfidf bigrams",    build_lr_pipeline(C=best_C, ngram_range=(1, 2))),
    ]
    rows = []
    for label, pipe in configs:
        pipe.fit(train_texts, train_labels)
        preds = pipe.predict(val_texts)
        report = classification_report(val_labels, preds, output_dict=True)
        rows.append({
            "model":        label.split("|")[0].strip(),
            "features":     label.split("|")[1].strip(),
            "val_macro_f1": round(report["macro avg"]["f1-score"], 4),
            "val_accuracy": round(report["accuracy"], 4),
        })
    return pd.DataFrame(rows).sort_values("val_macro_f1", ascending=False)


abl_df = full_ablation(train_texts, train_labels, val_texts, val_labels, best_C=best_C)
print(abl_df.to_string(index=False))


### Instructor notes — Part 5

Expected ordering (val macro F1):
1. LR tfidf bigrams: ~0.895–0.902
2. LR tfidf unigrams: ~0.889–0.895
3. NB tfidf bigrams: ~0.877–0.885
4. NB tfidf unigrams: ~0.870–0.878
5. NB count unigrams: ~0.845–0.860

Bigrams help LR slightly less than NB in relative terms because LR already
captures co-occurrence signals through the weight matrix.

Cost-efficiency sweet spot for most use cases: **LR tfidf unigrams** — nearly
as good as bigrams but 5-10x smaller feature space.


## Part 6 — Final test evaluation

In [ ]:
# ── Train best config on full train split, evaluate on held-out test ─────────
best_pipe = build_lr_pipeline(C=best_C, ngram_range=(1, 2))
best_pipe.fit(train_texts, train_labels)
test_pred = best_pipe.predict(test_texts)

test_results = evaluate_classifier(
    test_labels, test_pred, label_names,
    title=f"LR (TF-IDF bigrams, C={best_C}) — TEST SET",
)
print(f"Test macro F1: {test_results['macro_f1']}")


## Bonus — Gradient descent from scratch

Instructor note: the bonus is calibrated for students who finish parts 1-5 early.
Expected test: does the loss decrease monotonically? Is the decision boundary reasonable?


In [ ]:
# ── Synthetic 2D dataset ──────────────────────────────────────────────────────
rng = np.random.default_rng(SEED)
n_pos, n_neg = 200, 200
X_pos = rng.multivariate_normal([2, 2], [[1, 0.5], [0.5, 1]], n_pos)
X_neg = rng.multivariate_normal([-2, -2], [[1, -0.3], [-0.3, 1]], n_neg)

X = np.vstack([X_pos, X_neg]).astype(np.float64)
y = np.array([1] * n_pos + [0] * n_neg)

shuffle_idx = rng.permutation(len(y))
X, y = X[shuffle_idx], y[shuffle_idx]

X_bias = np.hstack([np.ones((len(X), 1)), X])
print(f"X shape: {X_bias.shape}, y shape: {y.shape}")


In [ ]:
def sigmoid(z: np.ndarray) -> np.ndarray:
    # ⚠️ STUDENT MISTAKE: naive implementation overflows for large |z|.
    # Use np.where or scipy.special.expit for numerical stability.
    return np.where(z >= 0,
                    1.0 / (1.0 + np.exp(-z)),
                    np.exp(z) / (1.0 + np.exp(z)))


def cross_entropy_loss(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    eps = 1e-15
    y_pred = np.clip(y_pred, eps, 1 - eps)
    return float(-np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred)))


def gradient_descent_lr(
    X: np.ndarray,
    y: np.ndarray,
    learning_rate: float = 0.1,
    n_epochs: int = 200,
) -> tuple:
    N, D = X.shape
    w = np.zeros(D)
    losses = []

    for _ in range(n_epochs):
        z    = X @ w
        y_hat = sigmoid(z)
        grad  = (X.T @ (y_hat - y)) / N      # shape (D,)
        w    -= learning_rate * grad
        losses.append(cross_entropy_loss(y, sigmoid(X @ w)))

    return w, losses


w_learned, losses = gradient_descent_lr(X_bias, y, learning_rate=0.1, n_epochs=200)

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(losses)
ax.set_xlabel("Epoch")
ax.set_ylabel("Cross-entropy loss")
ax.set_title("Training loss — gradient descent")
plt.tight_layout()
plt.show()

print(f"Final loss: {losses[-1]:.4f}")
print(f"Learned weights: bias={w_learned[0]:.3f}, w1={w_learned[1]:.3f}, w2={w_learned[2]:.3f}")


In [ ]:
def plot_decision_boundary(
    X: np.ndarray,
    y: np.ndarray,
    w: np.ndarray,
    title: str = "Logistic Regression decision boundary",
) -> None:
    x1_min, x1_max = X[:, 0].min() - 1, X[:, 0].max() + 1

    # Decision boundary: w[0] + w[1]*x1 + w[2]*x2 = 0 => x2 = -(w[0] + w[1]*x1) / w[2]
    x1_vals = np.linspace(x1_min, x1_max, 200)
    x2_vals = -(w[0] + w[1] * x1_vals) / w[2]

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(X[y == 1, 0], X[y == 1, 1], alpha=0.5, label="Class 1", color="#3a86ff", s=20)
    ax.scatter(X[y == 0, 0], X[y == 0, 1], alpha=0.5, label="Class 0", color="#ff595e",  s=20)
    ax.plot(x1_vals, x2_vals, "k-", linewidth=2, label="Decision boundary")
    ax.set_xlabel("x1"); ax.set_ylabel("x2")
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    plt.show()


plot_decision_boundary(X, y, w_learned)


## Summary — Running comparison table

In [ ]:
results_table = pd.DataFrame([
    {"session": "01", "model": "Majority baseline",    "features": "none",             "val_macro_f1": 0.25,  "test_macro_f1": 0.25},
    {"session": "03", "model": "NB (scratch)",         "features": "count unigrams",   "val_macro_f1": None,  "test_macro_f1": None},
    {"session": "03", "model": "NB (sklearn)",         "features": "count unigrams",   "val_macro_f1": None,  "test_macro_f1": None},
    {"session": "03", "model": "NB (sklearn)",         "features": "tfidf bigrams",    "val_macro_f1": None,  "test_macro_f1": None},
    {"session": "03", "model": "LR",                   "features": "tfidf unigrams",   "val_macro_f1": None,  "test_macro_f1": None},
    {"session": "03", "model": "LR",                   "features": "tfidf bigrams",    "val_macro_f1": test_results["macro_f1"], "test_macro_f1": test_results["macro_f1"]},
])
# Fill in from your actual runs above
print(results_table.to_string(index=False))
